In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
from tqdm import tqdm
import gc
from numba import njit
import duckdb

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df = pd.read_parquet('../data/study_datasets/question_centered_model_7d_processed.parquet')
print(df.columns.tolist())

['eventId', 'phase', 'userId', 'timestamp', 'event', 'questionId', 'phaseOneStart', 'phaseTwoEnd', 'hasAnswer', 'hasAcceptedAnswer', 'timeToFirstAnswerHours', 'timeToAcceptedAnswerHours', 'timeToAcceptVoteHours', 'numHelped', 'helpProvidedEver', 'receivedAcceptedAnswerEver', 'receivedAcceptedVoteEver', 'year', 'month', 'numQuestionsAskedAT', 'numHelpProvidedAT', 'numAcceptedAnswersReceivedAT', 'numAcceptedVotesReceivedAT', 'numQuestionsAsked30D', 'numHelpProvided30D', 'numAcceptedAnswersReceived30D', 'numAcceptedVotesReceived30D', 'numAcceptedAnswersPosted30D', 'numQuestionsAsked14D', 'numHelpProvided14D', 'numAcceptedAnswersReceived14D', 'numAcceptedVotesReceived14D', 'numAcceptedAnswersPosted14D', 'numQuestionsAsked7D', 'numHelpProvided7D', 'numAcceptedAnswersReceived7D', 'numAcceptedVotesReceived7D', 'numAcceptedAnswersPosted7D', 'numQuestionsAsked3D', 'numHelpProvided3D', 'numAcceptedAnswersReceived3D', 'numAcceptedVotesReceived3D', 'numAcceptedAnswersPosted3D', 'numAcceptedAnswers

# Basic Models with just phase interaction

In [2]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
sys.modules['stargazer.translators.statsmodels'].pd = pd

# Define the formula for the three models - fixed column names
formula1 = "numHelped ~ C(phase) + C(hasAnswer) + C(phase):C(hasAnswer)"
formula2 = "userFeNumHelped ~ C(phase) + C(hasAnswer) + C(phase):C(hasAnswer)"
formula3 = "questionFeNumHelped ~ C(phase) + C(hasAnswer) + C(phase):C(hasAnswer)"

# Model names
model_names = ["Base Model", "User FE", "Question FE"]
formulas = [formula1, formula2, formula3]

# Run and display each model individually
for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"\n\n==== {name} ====")

    # Fit the model
    model = smf.ols(formula=formula, data=df).fit()

    # Apply clustered standard errors by userId (fixed column name)
    model = model.get_robustcov_results(
        cov_type='cluster',
        groups=df['userId']
    )

    # Create a Stargazer table for this single model
    single_stargazer = Stargazer([model])
    single_stargazer.title(f"Effect of Receiving Answers on Providing Help - {name}")
    single_stargazer.significant_digits(3)
    single_stargazer.show_degrees_of_freedom(False)
    single_stargazer.show_model_numbers(False)

    # Display HTML output for this model
    html_output = single_stargazer.render_html()
    display(HTML(html_output))



==== Base Model ====




==== User FE ====




==== Question FE ====


C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '


# Models with numHelpProvided Buckets

In [3]:
# Distribution table showing users with at least X help up to 10
user_max_help = df.groupby('userId')['numHelpProvidedAT'].max().reset_index()
print("\nDistribution of Users with Max numHelpProvidedAT at least:")
total_users = len(user_max_help)
for threshold in range(11):  # 0 to 10
    users_with_at_least = (user_max_help['numHelpProvidedAT'] >= threshold).sum()
    percentage = users_with_at_least / total_users * 100
    print(f"  {threshold}: {users_with_at_least:,} users ({percentage:.1f}%)")

# Define cutoff and filter to users with max help >= cutoff
cutoff = 5
print(f"\nUsing a cutoff of numHelpProvidedAT >= {cutoff}")
included_users = user_max_help[user_max_help['numHelpProvidedAT'] >= cutoff]['userId'].tolist()
filtered_by_user_df = df[df['userId'].isin(included_users)].copy()
print(f"\nFiltered to {len(included_users):,} users who have provided at least {cutoff} help at least once")
print(f"Events after user filtering: {len(filtered_by_user_df):,} ({len(filtered_by_user_df)/len(df):.1%} of original data)")
filtered_df = filtered_by_user_df[filtered_by_user_df['numHelpProvidedAT'] <= cutoff].copy()
print(f"Events after max help filtering: {len(filtered_df):,} ({len(filtered_df)/len(filtered_by_user_df):.1%} of user-filtered data)")
filtered_df['numHelpProvidedAT_binned'] = filtered_df['numHelpProvidedAT']

# Re-calculate user FEs
filtered_df['userFeNumHelped'] = filtered_df.groupby("userId")["numHelped"].transform(lambda x: x - x.mean())
filtered_df['userFeHasHelped'] = filtered_df.groupby("userId")["hasHelped"].transform(lambda x: x - x.mean())
filtered_df['userFeLnNumHelped'] = filtered_df.groupby("userId")["lnNumHelped"].transform(lambda x: x - x.mean())

# Show distribution of events by bin
bin_counts = filtered_df['numHelpProvidedAT_binned'].value_counts().sort_index()
print("\nDistribution of numHelpProvidedAT_binned in filtered dataset:")
for bin_val, count in bin_counts.items():
    percentage = count / len(filtered_df) * 100
    print(f"  {bin_val}: {count:,} events ({percentage:.1f}%)")

# Define formulas with bin interactions (0 is reference level)
formula1 = """numHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)"""

formula2 = """userFeNumHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)"""

formula3 = """questionFeNumHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)"""

model_names = ["Base Model", "User FE", "Question FE"]
formulas = [formula1, formula2, formula3]

# Run and display each model with clustered standard errors
for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"\n\n==== {name} ====")
    print(f"Formula: {formula}")

    model = smf.ols(formula=formula, data=filtered_df).fit()
    model = model.get_robustcov_results(
        cov_type='cluster',
        groups=filtered_df['userId']
    )

    single_stargazer = Stargazer([model])
    single_stargazer.title(f"Effect of Receiving Answers on Providing Help - {name}")
    single_stargazer.significant_digits(3)
    single_stargazer.show_degrees_of_freedom(False)
    single_stargazer.show_model_numbers(False)

    html_output = single_stargazer.render_html()
    display(HTML(html_output))


Distribution of Users with Max numHelpProvidedAT at least:
  0: 99,969 users (100.0%)
  1: 18,835 users (18.8%)
  2: 13,209 users (13.2%)
  3: 10,688 users (10.7%)
  4: 9,262 users (9.3%)
  5: 8,166 users (8.2%)
  6: 7,361 users (7.4%)
  7: 6,734 users (6.7%)
  8: 6,188 users (6.2%)
  9: 5,728 users (5.7%)
  10: 5,302 users (5.3%)

Using a cutoff of numHelpProvidedAT >= 5

Filtered to 8,166 users who have provided at least 5 help at least once
Events after user filtering: 308,508 (33.0% of original data)
Events after max help filtering: 111,850 (36.3% of user-filtered data)

Distribution of numHelpProvidedAT_binned in filtered dataset:
  0: 42,458 events (38.0%)
  1: 16,444 events (14.7%)
  2: 13,680 events (12.2%)
  3: 11,982 events (10.7%)
  4: 12,032 events (10.8%)
  5: 15,254 events (13.6%)


==== Base Model ====
Formula: numHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)




==== User FE ====
Formula: userFeNumHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)




==== Question FE ====
Formula: questionFeNumHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)


C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 23, but rank is 12
  warnings.warn('covariance of constraints does not have full '


# Models with experience receiving/providing

In [4]:
import sys
import statsmodels.formula.api as smf
from IPython.display import HTML
import pandas as pd
import numpy as np
from stargazer.stargazer import Stargazer
sys.modules['stargazer.translators.statsmodels'].pd = pd

# Model 1: Users with both numQuestionsAskedAT=0 and numQuestionsAskedAT=1
# First identify users who have both values
users_with_0 = set(df[df['numQuestionsAskedAT'] == 0]['userId'])
users_with_1 = set(df[df['numQuestionsAskedAT'] == 1]['userId'])
users_with_both = users_with_0.intersection(users_with_1)

print(f"Users with both values: {len(users_with_both)}")

# Filter dataset to only include these users and where numQuestionsAskedAT is 0 or 1
filtered_df1 = df[
    (df['userId'].isin(users_with_both)) &
    (df['numQuestionsAskedAT'].isin([0, 1]))
].copy()

print(f"Distribution of initialExperienceReceiving values:")
print(filtered_df1['initialExperienceReceiving'].value_counts())

# Convert string variables to categorical type
filtered_df1['initialExperienceReceiving'] = filtered_df1['initialExperienceReceiving'].astype('category')
filtered_df1['hasAnswer'] = filtered_df1['hasAnswer'].astype(int)  # Make sure this is numeric

# Formula for Model 1
formula1 = 'questionFeNumHelped ~ phase * C(initialExperienceReceiving) + phase:C(initialExperienceReceiving):C(hasAnswer)'

# Fit Model 1
model1 = smf.ols(formula=formula1, data=filtered_df1).fit()

# Apply clustered standard errors by userId
model1 = model1.get_robustcov_results(
    cov_type='cluster',
    groups=filtered_df1['userId']
)

# Create Stargazer table for this model
model1_stargazer = Stargazer([model1])
model1_stargazer.title("Model 1: Effect of initialExperienceReceiving on Helping")
model1_stargazer.significant_digits(3)
model1_stargazer.show_degrees_of_freedom(False)
model1_stargazer.show_model_numbers(False)

# Display HTML output for Model 1
html_output1 = model1_stargazer.render_html()
display(HTML(html_output1))

# Model 2: Users with numQuestionsAskedAT > 0 and both numHelpProvidedAT=0 and numHelpProvidedAT=1
# First identify users who have both values of numHelpProvidedAT among those with numQuestionsAskedAT > 0
questioned_users = df[df['numQuestionsAskedAT'] > 0]['userId'].unique()
help0_users = set(df[(df['numQuestionsAskedAT'] > 0) & (df['numHelpProvidedAT'] == 0)]['userId'])
help1_users = set(df[(df['numQuestionsAskedAT'] > 0) & (df['numHelpProvidedAT'] == 1)]['userId'])
users_with_both_help = help0_users.intersection(help1_users)

print(f"Users with both values: {len(users_with_both_help)}")

# Filter dataset for Model 2
filtered_df2 = df[
    (df['userId'].isin(users_with_both_help)) &
    (df['numQuestionsAskedAT'] > 0) &
    (df['numHelpProvidedAT'].isin([0, 1]))
].copy()

print(f"Distribution of initialExperienceGiving values:")
print(filtered_df2['initialExperienceGiving'].value_counts())

# Convert string variables to categorical type for model 2
filtered_df2['initialExperienceGiving'] = filtered_df2['initialExperienceGiving'].astype('category')
filtered_df2['hasAnswer'] = filtered_df2['hasAnswer'].astype(int)

# Formula for Model 2
formula2 = 'questionFeNumHelped ~ phase * C(initialExperienceGiving) + phase:C(initialExperienceGiving):C(hasAnswer)'

# Fit Model 2
model2 = smf.ols(formula=formula2, data=filtered_df2).fit()

# Apply clustered standard errors by userId
model2 = model2.get_robustcov_results(
    cov_type='cluster',
    groups=filtered_df2['userId']
)

# Create Stargazer table for Model 2
model2_stargazer = Stargazer([model2])
model2_stargazer.title("Model 2: Effect of initialExperienceGiving on Helping")
model2_stargazer.significant_digits(3)
model2_stargazer.show_degrees_of_freedom(False)
model2_stargazer.show_model_numbers(False)

# Display HTML output for Model 2
html_output2 = model2_stargazer.render_html()
display(HTML(html_output2))

Users with both values: 38081
Distribution of initialExperienceReceiving values:
initialExperienceReceiving
no help seeked    81308
help seeked       47748
help received     42714
Name: count, dtype: Int64


Users with both values: 3945
Distribution of initialExperienceGiving values:
initialExperienceGiving
no help attempted    60860
help attempted       39626
helped                6494
Name: count, dtype: Int64
